# 📊 Análise de Churn em E-commerce

## 🎯 Objetivo
Identificar padrões de comportamento associados ao churn de usuários.

## 📥 Dados
Base de eventos de usuários (views, carts, purchases).

## 🧹 Tratamento
- Conversão de datas
- Criação de métricas por usuário
- Definição de churn (14 dias inativo)

## 📊 Análise Exploratória

## 💡 Insights


## 🚀 Conclusão
A análise demonstrou que o churn está fortemente relacionado ao nível de engajamento dos usuários.

Usuários mais ativos, que interagem com a plataforma e realizam compras, apresentam menor probabilidade de churn.

O principal gargalo identificado está na conversão inicial, especialmente na transição de visualização para adição ao carrinho.

Isso sugere que estratégias voltadas para aumentar o engajamento inicial e incentivar a primeira conversão podem ter impacto significativo na retenção de usuários.


In [8]:
import pandas as pd
import plotly.express as px
import ipykernel

In [9]:
df = pd.read_csv('/home/caoli/marketing-analytics-platform/data/raw/events.csv')

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42448764 entries, 0 to 42448763
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     str    
 1   event_type     str    
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  str    
 5   brand          str    
 6   price          float64
 7   user_id        int64  
 8   user_session   str    
dtypes: float64(1), int64(3), str(5)
memory usage: 2.8 GB


In [11]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [12]:
df['event_time'] = pd.to_datetime(df['event_time'])

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42448764 entries, 0 to 42448763
Data columns (total 9 columns):
 #   Column         Dtype              
---  ------         -----              
 0   event_time     datetime64[us, UTC]
 1   event_type     str                
 2   product_id     int64              
 3   category_id    int64              
 4   category_code  str                
 5   brand          str                
 6   price          float64            
 7   user_id        int64              
 8   user_session   str                
dtypes: datetime64[us, UTC](1), float64(1), int64(3), str(4)
memory usage: 2.8 GB


In [14]:
df['event_time'] = pd.to_datetime(df['event_time'])

In [15]:
last_activity = df.groupby('user_id')['event_time'].max().reset_index()

quando foi a última atividade de cada usuário?

In [16]:
reference_date = df['event_time'].max()

In [17]:
last_activity['days_inactive'] = (reference_date - last_activity['event_time']).dt.days

In [18]:
reference_date

Timestamp('2019-10-31 23:59:59+0000', tz='UTC')

In [19]:
last_activity.head()

,user_id,event_time,days_inactive
0,33869381,2019-10-23 20:04:08+00:00,8
1,64078358,2019-10-13 00:13:46+00:00,18
2,183503497,2019-10-02 21:43:00+00:00,29
3,184265397,2019-10-15 17:19:28+00:00,16
4,195082191,2019-10-10 03:35:36+00:00,21


In [20]:
df_small = df[['user_id', 'event_type', 'price', 'event_time']].copy()
df_small['event_type'] = df_small['event_type'].astype('category')

agg = (
    df_small.assign(
        is_view=(df_small['event_type'] == 'view').astype('int8'),
        is_cart=(df_small['event_type'] == 'cart').astype('int8'),
        is_purchase=(df_small['event_type'] == 'purchase').astype('int8')
    )
    .groupby('user_id')
    .agg(
        total_events=('event_type', 'count'),
        total_views=('is_view', 'sum'),
        total_carts=('is_cart', 'sum'),
        total_purchases=('is_purchase', 'sum'),
        avg_price=('price', 'mean')
    )
    .reset_index()
)

agg.head()

,user_id,total_events,total_views,total_carts,total_purchases,avg_price
0,33869381,1,1,0,0,769.650000
1,64078358,1,1,0,0,0.000000
2,183503497,1,1,0,0,15.770000
3,184265397,6,6,0,0,111.706667
4,195082191,1,1,0,0,161.880000


In [21]:
last_activity['churn'] = (last_activity['days_inactive'] > 14).astype(int)
last_activity['churn'].value_counts(normalize=True)

churn
0    0.62142
1    0.37858
Name: proportion, dtype: float64

In [22]:
final_df = agg.merge(last_activity[['user_id', 'churn']], on='user_id')
final_df.head()

,user_id,total_events,total_views,total_carts,total_purchases,avg_price,churn
0,33869381,1,1,0,0,769.650000,0
1,64078358,1,1,0,0,0.000000,1
2,183503497,1,1,0,0,15.770000,1
3,184265397,6,6,0,0,111.706667,1
4,195082191,1,1,0,0,161.880000,1


In [23]:
final_df['churn'].value_counts()

churn
0    1878110
1    1144180
Name: count, dtype: int64

quem churna compra menos? 

Usuários que churnaram apresentam menor número médio de compras, indicando baixa conversão ao longo da jornada.

In [24]:
final_df.groupby('churn')['total_purchases'].mean()

churn
0    0.314293
1    0.133347
Name: total_purchases, dtype: float64

quem só navega (view) churna mais

Usuários com maior volume de visualizações apresentam menor taxa de churn, indicando que engajamento está associado à retenção.

In [25]:
final_df.groupby('churn')['total_views'].mean()

churn
0    17.445115
1     7.005502
Name: total_views, dtype: float64

quem adiciona ao carrinho tende a churnar menos

A adição de produtos ao carrinho está associada a menor churn, indicando maior intenção de conversão.

In [26]:
final_df.groupby('churn')['total_carts'].mean()

churn
0    0.392491
1    0.165512
Name: total_carts, dtype: float64

In [27]:
import numpy as np

final_df['conversion_rate'] = np.where(
    final_df['total_views'] > 0,
    final_df['total_purchases'] / final_df['total_views'],
    0
)

In [28]:
final_df['no_purchase'] = (final_df['total_purchases'] == 0).astype(int)

final_df.groupby('churn')['no_purchase'].mean()

churn
0    0.865673
1    0.917113
Name: no_purchase, dtype: float64

Usuários que nunca realizaram compra apresentam maior propensão ao churn, reforçando a importância da primeira conversão.

Insights Principais
Usuários que churnaram apresentam menor número médio de compras, indicando baixa conversão ao longo da jornada.
Usuários mais engajados (com maior número de visualizações e interações) tendem a churnar menos.
A interação com o carrinho é um forte indicador de retenção, estando associada a menor churn.
A maioria dos usuários não realiza compras, mas a ausência de conversão está ainda mais presente entre usuários que churnam.
A primeira conversão (primeira compra) é um fator crítico para retenção de usuários.

In [30]:
import plotly.io as pio
pio.renderers.default = "browser"

In [31]:
import plotly.express as px

fig = px.box(
    final_df,
    x='churn',
    y='total_events',
    title='Atividade do Usuário vs Churn'
)

fig.show()

## 📊 Insight: Atividade vs Churn

Observa-se que usuários que não churnaram apresentam níveis significativamente mais altos de atividade, com maior dispersão e presença de outliers de alto engajamento.

Por outro lado, usuários que churnaram possuem comportamento concentrado em baixos níveis de interação, indicando menor engajamento com a plataforma.

Esse padrão sugere que o engajamento do usuário é um forte indicador de retenção, sendo usuários mais ativos menos propensos ao churn.

In [32]:
fig = px.box(
    final_df,
    x='churn',
    y='total_purchases',
    title='Compras vs Churn'
)

fig.show()

## 📊 Insight: Compras vs Churn

Observa-se que usuários que não churnaram apresentam maior volume de compras, com presença significativa de outliers de alto consumo.

Por outro lado, usuários que churnaram possuem comportamento concentrado em baixos níveis de compra, indicando baixa recorrência.

Esse padrão sugere que a frequência de compras é um forte indicador de retenção, sendo usuários com maior histórico de compras menos propensos ao churn.

In [34]:
import pandas as pd

funnel = pd.DataFrame({
    'stage': ['view', 'cart', 'purchase'],
    'count': [
        final_df['total_views'].sum(),
        final_df['total_carts'].sum(),
        final_df['total_purchases'].sum()
    ]
})

funnel

,stage,count
0,view,40779399
1,cart,926516
2,purchase,742849


In [35]:
import plotly.express as px

fig = px.funnel(
    funnel,
    x='count',
    y='stage',
    title='Funil de Conversão'
)

fig.show()

## 📊 Insight: Funil de Conversão

Observa-se uma queda extremamente significativa entre a etapa de visualização (view) e adição ao carrinho (cart), indicando baixa conversão inicial.

Esse comportamento sugere que, embora muitos usuários naveguem pela plataforma, poucos demonstram intenção real de compra.

A queda entre carrinho e compra também está presente, porém em menor intensidade, indicando que usuários que chegam ao carrinho já possuem maior propensão à conversão.

Dessa forma, o principal gargalo do processo está na transformação de interesse em intenção de compra, e não na finalização da compra em si.


recomendação de produtos
UX (experiência)
destaque de produtos
preço / percepção de valor

NEM entra no funil

In [36]:
df_churn = final_df[final_df['churn'] == 1]
df_no_churn = final_df[final_df['churn'] == 0]

In [38]:
import pandas as pd

funnel_compare = pd.DataFrame({
    'stage': ['view', 'cart', 'purchase'],
    'churn': [
        df_churn['total_views'].sum(),
        df_churn['total_carts'].sum(),
        df_churn['total_purchases'].sum()
    ],
    'no_churn': [
        df_no_churn['total_views'].sum(),
        df_no_churn['total_carts'].sum(),
        df_no_churn['total_purchases'].sum()
    ]
})

funnel_compare

,stage,churn,no_churn
0,view,8015555,32763844
1,cart,189375,737141
2,purchase,152573,590276


In [39]:
funnel_melted = funnel_compare.melt(
    id_vars='stage',
    var_name='group',
    value_name='count'
)

In [40]:
import plotly.express as px

fig = px.funnel(
    funnel_melted,
    x='count',
    y='stage',
    color='group',
    title='Funil de Conversão: Churn vs Não Churn'
)

fig.show()

## 📊 Insight: Funil por Churn

Ao comparar o funil de conversão entre usuários que churnaram e não churnaram, observa-se uma diferença significativa no comportamento.

Usuários que churnaram apresentam uma queda muito mais acentuada nas primeiras etapas do funil, especialmente entre visualização e adição ao carrinho.

Por outro lado, usuários que não churnaram percorrem mais etapas do funil e apresentam maior taxa de conversão.

Esse padrão reforça que o churn está diretamente relacionado ao baixo engajamento inicial e à falta de progressão no funil de conversão.